# 🏎️ F1 Pit Stop Prediction | XGBoost + Full Feature Engineering
### Playground Series S6E5

**XGBoost with Optuna-tuned hyperparameters | StratifiedKFold OOF | Rich Feature Engineering from RealMLP notebook**

- Full feature engineering pipeline (tyre physics, race progress flags, interactions)
- NN-style encoding layer (floor-to-cat, count encoding, quantile binning, combo cats)
- Target encoding on combo features **inside** each fold (no leakage)
- StratifiedKFold to preserve class balance across folds
- Original dataset concatenated inside each fold

## 1. Imports & Seed

In [21]:
import random
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(42)

## 2. Config

In [22]:
class CFG:
    FOLDS  = 5          # StratifiedKFold splits
    SEED   = 42
    TARGET = 'PitNextLap'
    ID     = 'id'

# ── Optuna-tuned XGBoost hyperparameters (from xgb-and-group-k-fold notebook) ──
XGB_PARAMS = {
    # Core hyperparameters
    "learning_rate"      : 0.005152269004557939,
    "max_depth"          : 10,
    "min_child_weight"   : 11.83503440474689,
    "subsample"          : 0.5144055150851956,
    "gamma"              : 0.17091899550489928,
    "colsample_bytree"   : 0.5059508549225745,
    "colsample_bylevel"  : 0.9945701925466703,
    "reg_lambda"         : 0.032776579651102616,
    "reg_alpha"          : 0.0155251922035826,
    "max_delta_step"     : 6.937126157478375,

    # Training settings
    "n_estimators"       : 30_000,
    "enable_categorical" : True,   # XGBoost native cat handling
    "objective"          : "binary:logistic",
    "eval_metric"        : "auc",
    "device"             : "cuda",  # change to "cpu" if no GPU
    "early_stopping_rounds": 500,
    "random_state"       : 42,
}

## 3. Load Data

In [23]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')
orig  = pd.read_csv('/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv')

orig.drop(columns=['Normalized_TyreLife'], inplace=True, errors='ignore')

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Orig  : {orig.shape}")
print(f"Target rate train : {train[CFG.TARGET].mean():.4f}")
print(f"Target rate orig  : {orig[CFG.TARGET].mean():.4f}")

Train : (439140, 16)
Test  : (188165, 15)
Orig  : (101371, 15)
Target rate train : 0.1990
Target rate orig  : 0.2548


## 4. Feature Engineering

Tyre physics + race-progress signals + interaction terms.

In [24]:
# Domain knowledge: median stint lengths per compound, and hardness scores
COMPOUND_STINT_MEDIANS = {
    'SOFT': 14.0, 'MEDIUM': 17.0, 'HARD': 23.0,
    'INTERMEDIATE': 16.0, 'WET': 14.0
}
COMPOUND_HARDNESS = {
    'SOFT': 1, 'MEDIUM': 2, 'HARD': 3,
    'INTERMEDIATE': 1, 'WET': 0
}

def engineer_features(df):
    df = df.copy()

    # ── Core tyre features ───────────────────────────────────────────────────
    # ExpectedStint: how long this compound typically lasts
    df['ExpectedStint']       = df['Compound'].map(COMPOUND_STINT_MEDIANS).fillna(17.0)
    # TyreLife_Normalized: 0→1 journey through compound's expected life
    df['TyreLife_Normalized'] = df['TyreLife'] / df['ExpectedStint']
    # Polynomial transforms to help models learn non-linear tyre wear
    df['TyreLife_sq']         = df['TyreLife'] ** 2
    df['TyreLife_sqrt']       = np.sqrt(df['TyreLife'])
    df['TyreLife_log1p']      = np.log1p(df['TyreLife'])
    # Hardness score lets model reason about compound durability numerically
    df['Compound_Hardness']   = df['Compound'].map(COMPOUND_HARDNESS).fillna(2).astype(int)
    # Cross-features: age weighted by compound hardness
    df['TyreLife_x_Hardness'] = df['TyreLife'] * df['Compound_Hardness']
    df['Norm_x_Hardness']     = df['TyreLife_Normalized'] * df['Compound_Hardness']

    # ── Tyre age flags ───────────────────────────────────────────────────────
    # Binary signals for key tyre age milestones
    df['Is_Fresh']      = (df['TyreLife'] <= 3).astype(np.int8)   # just pitted
    df['Is_Old']        = (df['TyreLife'] > 20).astype(np.int8)   # getting long
    df['Is_VeryOld']    = (df['TyreLife'] > 40).astype(np.int8)   # overdue
    df['Is_FirstStint'] = (df['Stint'] == 1).astype(np.int8)      # opening stint

    # ── Degradation rate ─────────────────────────────────────────────────────
    # Cumulative_Degradation per lap: how fast the tyre is wearing down
    df['DegRate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)

    # ── Race progress flags ──────────────────────────────────────────────────
    # Strategy windows: early/late pit pressure varies by race phase
    df['Early_Race']    = (df['RaceProgress'] < 0.25).astype(np.int8)
    df['Late_Race']     = (df['RaceProgress'] >= 0.75).astype(np.int8)
    df['VeryLate_Race'] = (df['RaceProgress'] >= 0.90).astype(np.int8)

    # ── Year flags ───────────────────────────────────────────────────────────
    df['Is_2023'] = (df['Year'] == 2023).astype(np.int8)
    df['Is_2025'] = (df['Year'] == 2025).astype(np.int8)

    # ── Ratio / interaction features ─────────────────────────────────────────
    df['TyreLife_per_Lap']   = df['TyreLife'] / df['LapNumber'].clip(lower=1)
    df['Lap_x_RaceProgress'] = df['LapNumber'] * df['RaceProgress']
    df['Stint_x_TyreLife']   = df['Stint'] * df['TyreLife']
    df['Stint_x_Normalized'] = df['Stint'] * df['TyreLife_Normalized']

    # ── LapTime signal ───────────────────────────────────────────────────────
    # Squared and absolute delta: slow laps signal tyre cliff or traffic
    df['LapDelta_sq']  = df['LapTime_Delta'] ** 2
    df['LapDelta_abs'] = df['LapTime_Delta'].abs()

    return df

train = engineer_features(train)
test  = engineer_features(test)
orig  = engineer_features(orig)

# Separate target before we modify orig further
y_orig = orig[CFG.TARGET].copy()
orig   = orig.drop(columns=[CFG.TARGET])

y        = train[CFG.TARGET].copy()
train_id = train[CFG.ID].copy()
test_id  = test[CFG.ID].copy()

X      = train.drop(columns=[CFG.ID, CFG.TARGET, 'ExpectedStint'])
X_test = test.drop(columns=[CFG.ID, 'ExpectedStint'])
orig   = orig.drop(columns=['ExpectedStint'], errors='ignore')

orig = orig.reindex(columns=X.columns, fill_value=0)

print(f"X      : {X.shape}")
print(f"X_test : {X_test.shape}")
print(f"orig   : {orig.shape}")

X      : (439140, 37)
X_test : (188165, 37)
orig   : (101371, 37)


## 5. Encoding Layer

Originally designed to give RealMLP a richer view of numerical data — but these encodings also help XGBoost:

| Encoding | What it does | Why it helps XGB |
|---|---|---|
| **Floor → category** | Discretises floats into integer buckets, stored as `category` dtype | Creates explicit split points; XGB's native cat handler finds optimal merges |
| **Count encoding** | Replaces driver/compound/race/year with their frequency | Gives model a proxy for how common/representative each group is |
| **Quantile binning** | 200-bin RaceProgress → ordinal category | Captures non-linear strategy phases without the model needing deep trees |
| **Combo categoricals** | Race×Compound, Race×Year, Driver×Compound concatenated | Explicit interaction terms — cheaper than relying on tree depth alone |
| **Target encoding (in-fold)** | Mean target per combo cat, fitted only on train fold | Powerful signal with zero leakage since it's computed inside CV |

In [25]:
category_map = {}
IMPORTANT_COMBOS = [('Race', 'Compound'), ('Race', 'Year'), ('Driver', 'Compound')]

# Numericals to floor-discretise into categories
NUM_COLS_BASE = ['LapNumber', 'Stint', 'TyreLife', 'Position',
                 'LapTime (s)', 'RaceProgress', 'Year', 'PitStop']

def feature_engineering_encoding(df, fit=False):
    df = df.copy()

    # ── Arithmetic interactions ──────────────────────────────────────────────
    df['_TyreLife_div_LapNumber'] = (
        df['TyreLife'] / df['LapNumber'].clip(lower=1)
    ).astype('float32')
    df['_LapNumber_div_RaceProgress'] = (
        df['LapNumber'] / (df['RaceProgress'] + 1e-6)
    ).astype('float32')

    extra_num = ['_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress']

    # ── Floor numericals → XGBoost category dtype ───────────────────────────
    # Flooring creates discrete buckets; XGB's enable_categorical handles merging
    for col in NUM_COLS_BASE + extra_num:
        cat_name = f"{col}_cat_"
        if col in df.columns:
            if fit:
                codes, uniques = np.floor(df[col]).factorize()
                category_map[col] = uniques
            else:
                uniques  = category_map.get(col, np.array([]))
                code_map = {cat: i for i, cat in enumerate(uniques)}
                codes    = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
            df[cat_name] = pd.Categorical(codes.astype(str))

    # ── Count encoding ───────────────────────────────────────────────────────
    for col in ['Driver', 'Compound', 'Race', 'Year']:
        count_name = f"_{col}_count"
        if col in df.columns:
            if fit:
                count_map = df[col].astype(str).value_counts()
                category_map[count_name] = count_map
            else:
                count_map = category_map.get(count_name, pd.Series(dtype=int))
            df[count_name] = df[col].astype(str).map(count_map).fillna(0).astype('int32')

    # ── RaceProgress quantile binning ────────────────────────────────────────
    # 200 bins captures fine-grained strategy windows without overfitting
    bin_name = 'RaceProgress_200_quantile_bin_'
    if fit:
        kb = KBinsDiscretizer(n_bins=200, encode='ordinal',
                              strategy='quantile', subsample=None)
        binned = kb.fit_transform(df[['RaceProgress']]).ravel().astype('int32')
        category_map[bin_name] = kb
    else:
        kb     = category_map.get(bin_name)
        binned = (kb.transform(df[['RaceProgress']]).ravel().astype('int32')
                  if kb else np.zeros(len(df), dtype='int32'))
    df[bin_name] = pd.Categorical(binned.astype(str))

    # ── Combo categorical features ───────────────────────────────────────────
    combo_names = []
    for cols in IMPORTANT_COMBOS:
        combo_name = '_'.join(cols) + '_combo_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques  = category_map.get(combo_name, np.array([]))
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes    = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = pd.Categorical(codes.astype(str))

    # Cast ALL non-numeric columns to category — XGBoost rejects object dtype
    for col in df.columns:
        if df[col].dtype == object or str(df[col].dtype) == 'string':
            df[col] = df[col].astype('category')

    return df, combo_names


def ensure_categorical(df):
    """Re-cast any object columns that sneak back in after pd.concat."""
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype('category')
    return df

def align_categories(df, reference):
    """
    For every category column, set its categories to match the reference df.
    XGBoost requires val/test categories to be a subset of train's categories.
    """
    for col in df.columns:
        if hasattr(df[col], 'cat') and col in reference.columns and hasattr(reference[col], 'cat'):
            df[col] = df[col].cat.set_categories(reference[col].cat.categories)
    return df


X,      combo_names = feature_engineering_encoding(X,      fit=True)
X_test, _           = feature_engineering_encoding(X_test, fit=False)
orig,   _           = feature_engineering_encoding(orig,   fit=False)

orig = orig.reindex(columns=X.columns, fill_value=0)
orig = ensure_categorical(orig)

print(f"X      after encoding : {X.shape}")
print(f"X_test after encoding : {X_test.shape}")
print(f"Combo features        : {combo_names}")
print(f"Object cols remaining : {list(X.select_dtypes('object').columns)}  ← should be []")

X      after encoding : (439140, 57)
X_test after encoding : (188165, 57)
Combo features        : ['Race_Compound_combo_', 'Race_Year_combo_', 'Driver_Compound_combo_']
Object cols remaining : []  ← should be []


## 6. StratifiedKFold OOF Training

**Why StratifiedKFold here vs GroupKFold in the other notebook?**
- `GroupKFold` (original XGB notebook) keeps entire races together → tests generalisation to *unseen race events*; conservative, less leakage risk
- `StratifiedKFold` (this notebook) keeps class balance across folds → better OOF calibration; matches what the RealMLP notebook uses; good for ensembling since OOF distributions align

For blending with RealMLP OOFs, StratifiedKFold is the right choice.

In [26]:
skf        = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
auc_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold}/{CFG.FOLDS}")
    print(f"{'='*55}")

    X_tr  = X.iloc[tr_idx].copy()
    y_tr  = y.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Add original dataset to this fold's training data
    X_tr = pd.concat([X_tr, orig.reset_index(drop=True)], axis=0).reset_index(drop=True)
    y_tr = pd.concat([y_tr, y_orig.reset_index(drop=True)], axis=0).reset_index(drop=True)
    # pd.concat can silently demote category → object; re-cast to keep XGB happy
    X_tr  = ensure_categorical(X_tr)
    X_val = ensure_categorical(X_val)
    X_tst = ensure_categorical(X_tst)
    # Align val/test category levels to train — XGBoost errors if val has unseen levels
    X_val = align_categories(X_val, X_tr)
    X_tst = align_categories(X_tst, X_tr)

    # ── Target encoding on combo features (inside fold → no leakage) ─────────
    te = TargetEncoder(cv=CFG.FOLDS, smooth='auto', shuffle=True, random_state=CFG.SEED)
    tr_enc  = te.fit_transform(X_tr[combo_names],  y_tr)
    val_enc = te.transform(X_val[combo_names])
    tst_enc = te.transform(X_tst[combo_names])

    te_names = [f"_{c}_TE" for c in combo_names]
    X_tr[te_names]  = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc
    # Final safety cast before handing to XGBoost
    X_tr  = ensure_categorical(X_tr)
    X_val = ensure_categorical(X_val)
    X_tst = ensure_categorical(X_tst)

    if fold == 1:
        print(f"Total features : {X_tr.shape[1]}")
        print(f"Train rows     : {len(X_tr):,}  (synthetic + original)")
        print(f"Val rows       : {len(X_val):,}")
        print(f"Pos rate train : {y_tr.mean():.4f}  |  val: {y_val.mean():.4f}")

    # ── Train XGBoost ─────────────────────────────────────────────────────────
    model = XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=500
    )

    val_preds            = model.predict_proba(X_val)[:, 1]
    fold_test_preds      = model.predict_proba(X_tst)[:, 1]

    oof_preds[val_idx]   = val_preds
    test_preds          += fold_test_preds / CFG.FOLDS

    fold_auc = roc_auc_score(y_val, val_preds)
    auc_scores.append(fold_auc)
    best_iter = model.best_iteration
    print(f"\nFold {fold} AUC: {fold_auc:.6f}  |  Best iteration: {best_iter}")


  FOLD 1/5
Total features : 60
Train rows     : 452,683  (synthetic + original)
Val rows       : 87,828
Pos rate train : 0.2115  |  val: 0.1990
[0]	validation_0-auc:0.93193
[500]	validation_0-auc:0.94906
[1000]	validation_0-auc:0.95014
[1500]	validation_0-auc:0.95076
[2000]	validation_0-auc:0.95119
[2500]	validation_0-auc:0.95143
[3000]	validation_0-auc:0.95161
[3500]	validation_0-auc:0.95170
[4000]	validation_0-auc:0.95175
[4500]	validation_0-auc:0.95179
[5000]	validation_0-auc:0.95181
[5373]	validation_0-auc:0.95178

Fold 1 AUC: 0.951813  |  Best iteration: 4873

  FOLD 2/5
[0]	validation_0-auc:0.92905
[500]	validation_0-auc:0.94732
[1000]	validation_0-auc:0.94834
[1500]	validation_0-auc:0.94889
[2000]	validation_0-auc:0.94925
[2500]	validation_0-auc:0.94946
[3000]	validation_0-auc:0.94961
[3500]	validation_0-auc:0.94968
[4000]	validation_0-auc:0.94971
[4500]	validation_0-auc:0.94973
[5000]	validation_0-auc:0.94973
[5500]	validation_0-auc:0.94972
[5596]	validation_0-auc:0.94971

Fol

## 7. Results

In [27]:
oof_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*45}")
print(f"  Per-fold AUCs : {[f'{s:.4f}' for s in auc_scores]}")
print(f"  Mean fold AUC : {np.mean(auc_scores):.6f}")
print(f"  OOF ROC-AUC   : {oof_auc:.6f}")
print(f"{'='*45}")


  Per-fold AUCs : ['0.9518', '0.9498', '0.9507', '0.9499', '0.9516']
  Mean fold AUC : 0.950761
  OOF ROC-AUC   : 0.950752


## 8. Save OOF + Submission

In [28]:
oof_df = pd.DataFrame({
    CFG.ID    : train_id.values,
    CFG.TARGET: oof_preds,
})
oof_df.to_csv('oof_xgb_fe.csv', index=False)
print(f"OOF saved: oof_xgb_fe.csv  ({len(oof_df):,} rows)")

sub = pd.DataFrame({
    CFG.ID    : test_id.values,
    CFG.TARGET: test_preds,
})
sub.to_csv('submission_xgb_fe.csv', index=False)
print(f"Submission saved: submission_xgb_fe.csv")
print(sub.head())

OOF saved: oof_xgb_fe.csv  (439,140 rows)
Submission saved: submission_xgb_fe.csv
       id  PitNextLap
0  439140    0.005072
1  439141    0.005140
2  439142    0.004693
3  439143    0.159434
4  439144    0.839200


## Summary

| Component | Choice | Rationale |
|---|---|---|
| **Hyperparameters** | Optuna-tuned from xgb notebook | Low LR (0.005) + 30k trees + early stopping → well-regularised |
| **CV strategy** | StratifiedKFold 5-fold | Preserves class balance; OOF aligns with RealMLP for blending |
| **Feature engineering** | Full tyre physics + race-progress | Domain-aware features the model can directly split on |
| **Encoding layer** | Floor-cat + count + quantile bin + combos | Gives XGB structured views that typically need deep trees to discover |
| **Target encoding** | In-fold on combo cats | Powerful mean-target signal with zero train→val leakage |
| **Original data** | Appended inside each fold | More signal, especially for rare race/compound combos |

**Blend this OOF with the RealMLP OOF** — both use the same StratifiedKFold split indices so the predictions are directly stackable.